# LimiX Classification Example

This notebook demonstrates how to use the FAIM SDK's **TabularClient** with **LimiX** for tabular classification tasks.

LimiX is a foundation model for tabular machine learning that supports both classification and regression.

## Setup

Install dependencies and import required libraries.

In [6]:
import os

import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

from faim_sdk import LimiXPredictRequest, TabularClient

## Load and Prepare Data

Load the breast cancer classification dataset from scikit-learn.

In [7]:
# Load breast cancer dataset
X, y = load_breast_cancer(return_X_y=True)

# Split with 50/50 train-test split for demonstration
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

# Convert to float32 for API
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test = y_test.astype(np.float32)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Classes: {np.unique(y_train)}")

Training set size: (284, 30)
Test set size: (285, 30)
Number of features: 30
Classes: [0. 1.]


## Initialize TabularClient

Create a client to interact with the LimiX model.

In [8]:
# Initialize the client
client = TabularClient(
    base_url="https://api.faim.it.com",
    api_key=os.environ.get("FAIM_API_KEY"),  # Replace with your actual API key
    timeout=120.0,
)

print("TabularClient initialized!")

TabularClient initialized!


## Create Classification Request

Prepare a LimiX classification request.

In [9]:
# Create a LimiX classification request
request = LimiXPredictRequest(
    X_train=X_train, y_train=y_train, X_test=X_test, task_type="Classification", use_retrieval=False, model_version="1"
)

print("Request prepared:")
print(f"  X_train shape: {request.X_train.shape}")
print(f"  X_test shape: {request.X_test.shape}")
print(f"  Task type: {request.task_type}")

Request prepared:
  X_train shape: (284, 30)
  X_test shape: (285, 30)
  Task type: Classification


## Make Predictions

Send the request to LimiX and get classification predictions.

In [10]:
try:
    # Make predictions
    response = client.predict(request)

    print(f"Predictions shape: {response.predictions.shape}")
    print(f"First 10 predictions: {response.predictions[:10]}")

    if response.probabilities is not None:
        print(f"\nClass probabilities shape: {response.probabilities.shape}")
        print(f"First 3 samples probabilities:\n{response.probabilities[:3]}")

    print("\nMetadata:")
    for key, value in response.metadata.items():
        print(f"  {key}: {value}")

except Exception as e:
    print(f"Error: {e}")
    print("\nMake sure your API key is valid and the service is available.")

Request serialization failed
Traceback (most recent call last):
  File "/Users/andreichernov/Documents/Personal/research/FAIM/faim-client/faim_sdk/tabular_client.py", line 184, in predict
    payload = serialize_to_arrow(arrays, metadata, compression=request.compression)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/andreichernov/Documents/Personal/research/FAIM/faim-client/faim_sdk/utils.py", line 78, in serialize_to_arrow
    batch = pa.record_batch(cols, schema=schema)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "pyarrow/table.pxi", line 6045, in pyarrow.lib.record_batch
  File "pyarrow/table.pxi", line 3552, in pyarrow.lib.RecordBatch.from_arrays
ValueError: Arrays were not all the same length: 8520 vs 8550


Error: Failed to serialize request: Arrays were not all the same length: 8520 vs 8550 (details: {'model': 'limix', 'error': 'Arrays were not all the same length: 8520 vs 8550'})

Make sure your API key is valid and the service is available.


## Evaluate Results

Calculate classification metrics.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

try:
    y_pred = response.predictions.astype(int)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("Classification Metrics:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
except NameError:
    print("Run prediction cell first to evaluate metrics.")

## Using Retrieval-Augmented Inference

Enable RAI for potentially better accuracy on small datasets.

In [ ]:
# Request with retrieval-augmented inference
request_with_rai = LimiXPredictRequest(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    task_type="Classification",
    use_retrieval=True,  # Enable RAI
    model_version="1",
)

print("Request with RAI enabled prepared.")
# response_rai = client.predict(request_with_rai)

## Error Handling

Handle specific exceptions from the SDK.

In [ ]:
from faim_sdk import AuthenticationError, ErrorCode, ValidationError


def make_prediction_with_error_handling():
    try:
        request = LimiXPredictRequest(X_train=X_train, y_train=y_train, X_test=X_test, task_type="Classification")
        response = client.predict(request)
        return response

    except ValidationError as e:
        print(f"Validation Error: {e.message}")
        if e.error_code == ErrorCode.INVALID_SHAPE:
            print("Check array shapes.")
        print(f"Request ID: {e.error_response.request_id}")

    except AuthenticationError as e:
        print(f"Auth Error: {e.message}")
        print("Check your API key.")


# Test error handling
# make_prediction_with_error_handling()